# Atmospheric Blocking Metrics Demo

This notebook demonstrates the PCMDI Metrics Package atmospheric blocking detection and analysis workflow.

**Features:**
- Deep learning-based blocking detection using U-Net/ResNet152
- Regional and seasonal blocking statistics
- Multiple output formats (NetCDF, CSV, NPY, JSON)
- Batch processing support

**References:**
- Valkonen, E., Lee, J., et al. (2023). U-Net based atmospheric blocking detection for climate model evaluation. *In preparation.*
- Wiedenmann et al. (2002). The climatology of blocking anticyclones. *J. Climate*, 15(23), 3459-3473.

## 1. Setup and Installation

### Requirements

```bash
# Install PMP with blocking support
conda install -c conda-forge pcmdi_metrics 'tensorflow>=2'

# Or with pip
pip install 'pcmdi_metrics[blocking]'

# Download pre-trained model (246 MB)
download_blocking_model.py
```

### Check Installation

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # Suppress TensorFlow warnings

# Check if TensorFlow is available
try:
    import tensorflow as tf
    print(f"✓ TensorFlow {tf.__version__} available")
    print(f"  GPU available: {tf.config.list_physical_devices('GPU')}")
except ImportError:
    print("✗ TensorFlow not installed")
    print("  Install with: conda install -c conda-forge 'tensorflow>=2'")

# Try importing blocking module
try:
    from pcmdi_metrics.blocking import (
        detect_blocking,
        compute_blocking_metrics,
        metrics_to_json,
        validate_tensorflow
    )
    print("✓ Blocking metrics module loaded")
    
    # Check TensorFlow through PMP
    has_tf = validate_tensorflow()
    print(f"  TensorFlow validated: {has_tf}")
    
except ImportError as e:
    print(f"✗ Blocking metrics unavailable: {e}")
    print("\nThis notebook requires TensorFlow. Install it to continue.")

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print("✓ All imports successful")

## 2. Load Sample Data

Load 500 hPa geopotential height data. The blocking detection requires:
- **Variable**: Geopotential height (zg)
- **Level**: 500 hPa (50000 Pa)
- **Frequency**: Daily
- **Grid**: Any horizontal grid (will be regridded to 2°×2° automatically)

In [ ]:
# Example: Load CMIP6 data
# Replace with your actual data path
data_path = 'path/to/zg_day_*.nc'

# For demo purposes, we'll create synthetic data
# In practice, replace this with your actual CMIP6 data

def create_demo_data(n_days=365):
    """Create synthetic Z500 data for demonstration."""
    print(f"Creating synthetic Z500 data ({n_days} days)...")
    
    # Coordinates
    lats = np.arange(-88, 90, 2)  # 2° grid
    lons = np.arange(0, 360, 2)
    times = pd.date_range('2000-01-01', periods=n_days, freq='D')
    
    # Base state with realistic Z500 values
    lat_gradient = 5400 + 400 * np.cos(np.deg2rad(lats))
    base_state = np.tile(lat_gradient[:, np.newaxis], (1, len(lons)))
    
    # Add synoptic variability
    np.random.seed(42)
    zg_data = np.zeros((n_days, len(lats), len(lons)))
    
    for t in range(n_days):
        perturbation = 200 * np.random.randn(len(lats), len(lons))
        
        # Smooth (blocking is large-scale)
        from scipy.ndimage import gaussian_filter
        perturbation = gaussian_filter(perturbation, sigma=3)
        
        zg_data[t] = base_state + perturbation
        
        # Add persistent blocking pattern
        if 30 <= t < 50:  # 20-day blocking episode
            # Northern Pacific blocking
            zg_data[t, 45:55, 90:100] += 300  # 40-50°N, 180-200°E
    
    # Create xarray Dataset
    ds = xr.Dataset(
        {
            'zg': (['time', 'lat', 'lon'], zg_data, {
                'long_name': 'Geopotential Height',
                'units': 'm',
                'level': '500 hPa'
            })
        },
        coords={
            'time': times,
            'lat': ('lat', lats, {'units': 'degrees_north'}),
            'lon': ('lon', lons, {'units': 'degrees_east'})
        },
        attrs={
            'source': 'Synthetic data for demonstration',
            'model': 'DEMO-MODEL'
        }
    )
    
    print(f"  Created: {zg_data.shape[0]} days, {zg_data.shape[1]}×{zg_data.shape[2]} grid")
    print(f"  Z500 range: {zg_data.min():.1f} - {zg_data.max():.1f} m")
    
    return ds

# Create or load data
ds = create_demo_data(n_days=365)

# Display dataset info
print("\nDataset:")
print(ds)

In [ ]:
# Quick visualization of input data
fig = plt.figure(figsize=(14, 5))

# Map view
ax1 = plt.subplot(1, 2, 1, projection=ccrs.PlateCarree(central_longitude=180))
ds['zg'].isel(time=40).plot(
    ax=ax1,
    transform=ccrs.PlateCarree(),
    cmap='RdYlBu_r',
    cbar_kwargs={'label': 'Z500 (m)', 'shrink': 0.8}
)
ax1.coastlines()
ax1.add_feature(cfeature.BORDERS, linestyle=':')
ax1.set_title('500 hPa Geopotential Height (Day 40)', fontsize=12)

# Time series at a point
ax2 = plt.subplot(1, 2, 2)
ds['zg'].sel(lat=50, lon=180, method='nearest').plot(ax=ax2)
ax2.axvspan(ds.time[30], ds.time[49], alpha=0.2, color='red', label='Blocking episode')
ax2.set_title('Z500 Time Series (50°N, 180°E)', fontsize=12)
ax2.set_ylabel('Z500 (m)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Blocking Detection

Run neural network-based blocking detection.

**Process:**
1. Automatically regrid to 2°×2° if needed
2. Per-model normalization (zero mean, unit variance)
3. Create 5-day temporal windows (t-2, t-1, t, t+1, t+2)
4. Run U-Net/ResNet152 neural network
5. Apply threshold (default 0.34) for binary mask

In [ ]:
# Run blocking detection
print("Running blocking detection...")
print("This may take 30-60 seconds for 1 year of daily data.\n")

blocking = detect_blocking(
    ds,
    var='zg',
    level=50000,  # 500 hPa in Pa
    threshold=0.34,  # Optimized threshold
    output_file='demo_blocking_diagnostics.nc',  # Save diagnostics
    verbose=True
)

print("\nDetection complete!")
print(f"Output shape: {blocking['pval_binary'].shape}")
print(f"Note: 4 days shorter (2 removed from each end for 5-day window)")

In [ ]:
# Examine detection output
print("Blocking Detection Results:")
print(blocking)
print("\nVariables:")
print("  pval: Blocking probability [0-1]")
print("  pval_binary: Binary blocking mask (0=no block, 1=block)")

## 4. Visualize Blocking Detection

In [ ]:
# Plot blocking detection results
fig = plt.figure(figsize=(16, 10))

# 1. Single day - Probability
ax1 = plt.subplot(3, 2, 1, projection=ccrs.PlateCarree(central_longitude=180))
blocking['pval'].isel(time=38).plot(
    ax=ax1,
    transform=ccrs.PlateCarree(),
    cmap='Reds',
    vmin=0, vmax=1,
    cbar_kwargs={'label': 'Probability', 'shrink': 0.7}
)
ax1.coastlines()
ax1.set_title('Blocking Probability (Day 40)', fontsize=11)

# 2. Single day - Binary
ax2 = plt.subplot(3, 2, 2, projection=ccrs.PlateCarree(central_longitude=180))
blocking['pval_binary'].isel(time=38).plot(
    ax=ax2,
    transform=ccrs.PlateCarree(),
    cmap='RdYlGn_r',
    levels=[0, 0.5, 1],
    cbar_kwargs={'label': 'Blocking', 'ticks': [0.25, 0.75], 'shrink': 0.7}
)
ax2.coastlines()
ax2.set_title('Blocking Detection (Day 40)', fontsize=11)

# 3. Mean blocking frequency (all data)
ax3 = plt.subplot(3, 2, 3, projection=ccrs.PlateCarree(central_longitude=180))
blocking_freq = blocking['pval_binary'].mean(dim='time')
blocking_freq.plot(
    ax=ax3,
    transform=ccrs.PlateCarree(),
    cmap='YlOrRd',
    vmin=0, vmax=0.3,
    cbar_kwargs={'label': 'Frequency', 'shrink': 0.7}
)
ax3.coastlines()
ax3.set_title('Mean Blocking Frequency (All Days)', fontsize=11)

# 4. Zonal mean
ax4 = plt.subplot(3, 2, 4)
blocking_freq.mean(dim='lon').plot(ax=ax4, label='Annual mean')
ax4.axhline(0.1, color='gray', linestyle='--', alpha=0.5, label='10% threshold')
ax4.set_xlabel('Latitude')
ax4.set_ylabel('Blocking Frequency')
ax4.set_title('Zonal Mean Blocking Frequency', fontsize=11)
ax4.grid(alpha=0.3)
ax4.legend()

# 5. Time series of total blocked area
ax5 = plt.subplot(3, 2, 5)
blocked_area = blocking['pval_binary'].sum(dim=['lat', 'lon'])
blocked_area.plot(ax=ax5, marker='.')
ax5.axvspan(blocking.time[28], blocking.time[47], alpha=0.2, color='red')
ax5.set_ylabel('Blocked Grid Points')
ax5.set_title('Total Blocked Area Over Time', fontsize=11)
ax5.grid(alpha=0.3)

# 6. Latitude-time plot (Hovmöller)
ax6 = plt.subplot(3, 2, 6)
blocking['pval_binary'].sel(lon=180, method='nearest').T.plot(
    ax=ax6,
    cmap='RdYlGn_r',
    cbar_kwargs={'label': 'Blocking', 'shrink': 0.7}
)
ax6.set_title('Blocking Hovmöller (180°E)', fontsize=11)
ax6.set_ylabel('Latitude')

plt.tight_layout()
plt.show()

print(f"\nBlocking Statistics:")
print(f"  Total blocked points: {int(blocking['pval_binary'].sum().values)}")
print(f"  Blocking frequency: {float(blocking['pval_binary'].mean().values):.4f}")
print(f"  Days with blocking: {int((blocking['pval_binary'].sum(dim=['lat','lon']) > 0).sum().values)}")

## 5. Compute Regional and Seasonal Metrics

Calculate blocking statistics by region and season.

**Default Regions** (Wiedenmann et al. 2002):
- **NH**: Continental (30-60°N, 60-90°E), Pacific (30-60°N, 150°E-150°W), Atlantic (30-60°N, 90°W-30°E)
- **SH**: Indian (30-60°S, 20-90°E), Pacific (30-60°S, 90°E-150°W), Atlantic (30-60°S, 150-30°W)

**Seasons**: DJF, MAM, JJA, SON

In [ ]:
# Compute blocking metrics
print("Computing regional and seasonal metrics...\n")

metrics_df, spatial_means = compute_blocking_metrics(
    blocking,
    var='pval_binary',
    historical=True,  # Use historical period (1979-2010)
    verbose=True
)

print("\nMetrics computed successfully!")
print(f"  DataFrame shape: {metrics_df.shape}")
print(f"  Spatial means shape: {spatial_means.shape} (seasons × lat × lon)")

In [ ]:
# Display metrics table
print("\n" + "="*70)
print("BLOCKING METRICS BY REGION AND SEASON")
print("="*70)

# Show all metrics
pd.set_option('display.max_rows', None)
print(metrics_df.to_string(index=False))
pd.reset_option('display.max_rows')

print("\nColumns:")
print("  Year: Calendar year")
print("  Season: DJF (Dec-Jan-Feb), MAM (Mar-Apr-May), JJA (Jun-Jul-Aug), SON (Sep-Oct-Nov)")
print("  Region: Geographic region (6 regions total)")
print("  Number: Count of distinct blocking events (connected components)")
print("  Size: Mean size of blocking events (grid points)")

In [ ]:
# Summary statistics by season
print("\nSUMMARY BY SEASON:")
print("="*70)

summary = metrics_df.groupby('Season')[['Number', 'Size']].agg(['mean', 'std', 'min', 'max'])
print(summary)

# Summary by region
print("\nSUMMARY BY REGION:")
print("="*70)

summary_region = metrics_df.groupby('Region')[['Number', 'Size']].agg(['mean', 'std'])
print(summary_region)

In [ ]:
# Visualize metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Event count by season
ax1 = axes[0, 0]
season_counts = metrics_df.groupby('Season')['Number'].sum()
season_counts.plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Total Blocking Events by Season', fontsize=12, fontweight='bold')
ax1.set_xlabel('Season')
ax1.set_ylabel('Number of Events')
ax1.grid(axis='y', alpha=0.3)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)

# 2. Event count by region
ax2 = axes[0, 1]
region_counts = metrics_df.groupby('Region')['Number'].sum().sort_values(ascending=True)
region_counts.plot(kind='barh', ax=ax2, color='coral')
ax2.set_title('Total Blocking Events by Region', fontsize=12, fontweight='bold')
ax2.set_xlabel('Number of Events')
ax2.grid(axis='x', alpha=0.3)

# 3. Mean event size by season
ax3 = axes[1, 0]
metrics_df[metrics_df['Size'] > 0].boxplot(column='Size', by='Season', ax=ax3)
ax3.set_title('Blocking Event Size Distribution by Season', fontsize=12, fontweight='bold')
ax3.set_xlabel('Season')
ax3.set_ylabel('Event Size (grid points)')
plt.sca(ax3)
plt.xticks(rotation=0)

# 4. Seasonal cycle
ax4 = axes[1, 1]
for region in metrics_df['Region'].unique():
    region_data = metrics_df[metrics_df['Region'] == region]
    season_mean = region_data.groupby('Season')['Number'].mean()
    # Order seasons correctly
    season_order = ['DJF', 'MAM', 'JJA', 'SON']
    season_mean = season_mean.reindex(season_order)
    ax4.plot(season_order, season_mean, marker='o', label=region, linewidth=2)

ax4.set_title('Seasonal Cycle by Region', fontsize=12, fontweight='bold')
ax4.set_xlabel('Season')
ax4.set_ylabel('Mean Number of Events')
ax4.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Visualize Spatial Means by Season

In [ ]:
# Plot seasonal blocking frequency
seasons = ['DJF', 'MAM', 'JJA', 'SON']

fig = plt.figure(figsize=(16, 10))

for i, season in enumerate(seasons):
    ax = plt.subplot(2, 2, i+1, projection=ccrs.PlateCarree(central_longitude=180))
    
    # Plot seasonal mean
    seasonal_data = xr.DataArray(
        spatial_means[i],
        dims=['lat', 'lon'],
        coords={'lat': blocking.lat, 'lon': blocking.lon}
    )
    
    im = seasonal_data.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='YlOrRd',
        vmin=0,
        vmax=0.3,
        add_colorbar=True,
        cbar_kwargs={'label': 'Blocking Frequency', 'shrink': 0.8}
    )
    
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.5)
    
    # Add gridlines
    gl = ax.gridlines(draw_labels=True, alpha=0.3, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    
    ax.set_title(f'{season} Blocking Frequency', fontsize=13, fontweight='bold')

plt.suptitle('Seasonal Blocking Frequency', fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 7. Export Results

Save metrics in multiple formats:
- **NetCDF**: Full blocking detection data
- **CSV**: Metrics table
- **NPY**: Seasonal spatial means
- **JSON**: PCMDI-standard metrics format

In [ ]:
from pcmdi_metrics.blocking import metrics_to_csv, spatial_means_to_npy

# 1. Diagnostics already saved during detection
print("Output files:")
print("  1. demo_blocking_diagnostics.nc (already saved during detection)")

# 2. Save metrics to CSV
csv_file = 'demo_blocking_metrics.csv'
metrics_to_csv(metrics_df, csv_file)
print(f"  2. {csv_file} (metrics table)")

# 3. Save spatial means to NPY
npy_file = 'demo_blocking_spatial_means.npy'
spatial_means_to_npy(spatial_means, npy_file)
print(f"  3. {npy_file} (seasonal spatial means)")

# 4. Save to PCMDI JSON format
json_file = 'demo_blocking_metrics.json'
metrics_to_json(
    metrics_df,
    output_file=json_file,
    model_name='DEMO-MODEL',
    experiment='historical',
    reference_name='Synthetic',
    comment='Demonstration of atmospheric blocking metrics'
)
print(f"  4. {json_file} (PCMDI-standard JSON format)")

print("\n✓ All outputs saved successfully!")

In [ ]:
# Preview JSON structure
import json

with open(json_file) as f:
    json_data = json.load(f)

print("PCMDI JSON Structure:")
print("="*70)
print(f"Keys: {list(json_data.keys())}")
print(f"\nDIMENSIONS: {json_data['DIMENSIONS']}")
print(f"\nSample RESULTS structure:")
print(f"  Model → Region → Statistic → Season → Value")
print(f"\nPROVENANCE keys: {list(json_data['PROVENANCE'].keys())}")

## 8. Custom Regions (Advanced)

Define your own blocking regions for analysis.

In [ ]:
# Define custom regions
custom_regions = {
    'North_America': {
        'lat': slice(40, 65),
        'lon': slice(220, 290)  # 140°W to 70°W
    },
    'Europe': {
        'lat': slice(40, 70),
        'lon': slice(350, 40)  # 10°W to 40°E
    },
    'East_Asia': {
        'lat': slice(30, 60),
        'lon': slice(100, 150)  # 100°E to 150°E
    }
}

# Compute metrics with custom regions
print("Computing metrics with custom regions...\n")

custom_metrics, custom_spatial = compute_blocking_metrics(
    blocking,
    var='pval_binary',
    historical=True,
    regions=custom_regions,
    verbose=False
)

print("\nCustom Region Metrics:")
print(custom_metrics.groupby(['Season', 'Region'])[['Number', 'Size']].mean())

## 9. Batch Processing

For processing multiple models, use the operational driver:

In [ ]:
%%bash
# Create parameter file for batch processing
cat > demo_blocking_params.py << 'EOF'
# Batch processing parameters

# Experiments
exps = ["historical"]

# Data
var = "zg"
freq = "day"
cmipTable = "day"
mip_era = "CMIP6"

# Processing
first_member_only = True
parallel = True
num_processes = 4

# Output
result_dir = "output"
overwrite_output = False

# Debug mode (3 models, 2000-2010)
debug = False
EOF

echo "Parameter file created: demo_blocking_params.py"

In [ ]:
# Run batch processing (commented out - uncomment to run)
# !python -m pcmdi_metrics.blocking.blocking_driver --param demo_blocking_params.py

print("To run batch processing:")
print("  python -m pcmdi_metrics.blocking.blocking_driver --param demo_blocking_params.py")
print("\nOr in Python:")
print("  from pcmdi_metrics.blocking.blocking_driver import main")
print("  main(param_file='demo_blocking_params.py')")

## Summary

This notebook demonstrated:

1. ✓ **Installation** and dependency checking
2. ✓ **Loading data** (or creating synthetic demo data)
3. ✓ **Blocking detection** with neural network
4. ✓ **Visualization** of detection results
5. ✓ **Regional/seasonal metrics** calculation
6. ✓ **Multiple output formats** (NetCDF, CSV, NPY, JSON)
7. ✓ **Custom regions** for analysis
8. ✓ **Batch processing** setup

**Next Steps:**
- Apply to your own CMIP6 data
- Compare multiple models
- Validate against reanalysis (ERA5, MERRA2)
- Analyze future projections (SSP scenarios)

**Documentation:**
- Module README: `pcmdi_metrics/blocking/README.md`
- Model details: `pcmdi_metrics/blocking/data/README_model.md`
- API reference: `help(detect_blocking)`

**References:**
- Valkonen, E., Lee, J., et al. (2023). U-Net based atmospheric blocking detection for climate model evaluation. *In preparation.*
- Wiedenmann et al. (2002). The climatology of blocking anticyclones. *J. Climate*, 15(23), 3459-3473.